In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

# Dataset

## CVE data

In [3]:
import json
from tqdm import tqdm


cve_metadata = None
with open(f'../.data/cpe_json/nvdcve-1.1-2002.json', 'r') as f:
    cve_metadata = json.load(f)
for year in tqdm(range(2003, 2024+1)):
    with open(f'../.data/cpe_json/nvdcve-1.1-{year}.json', 'r') as f:
        cve_metadata["CVE_Items"].extend(json.load(f)["CVE_Items"])

100%|██████████| 22/22 [00:15<00:00,  1.40it/s]


In [4]:
sample_cve = cve_metadata["CVE_Items"][0]
print(sample_cve["publishedDate"])

id = sample_cve["cve"]["CVE_data_meta"]["ID"]
print(id)
desc = sample_cve["cve"]["description"]["description_data"][0]["value"]
print(desc)

sample_cve["cve"]["references"]["reference_data"]

1999-12-30T05:00Z
CVE-1999-0001
ip_input.c in BSD-derived TCP/IP implementations allows remote attackers to cause a denial of service (crash or hang) via crafted packets.


[{'url': 'http://www.openbsd.org/errata23.html#tcpfix',
  'name': 'http://www.openbsd.org/errata23.html#tcpfix',
  'refsource': 'CONFIRM',
  'tags': []},
 {'url': 'http://www.osvdb.org/5707',
  'name': '5707',
  'refsource': 'OSVDB',
  'tags': []}]

## Load VulFileHunter data

### Ground thruth data

In [5]:
import pickle

with open('../data/ground_truth/cve_github_can.pkl', 'rb') as f:
#with open('../data/ground_truth/ground_truth.pkl', 'rb') as f:
    ground_truth = pickle.load(f)


In [6]:
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

def normalize_url(url):
    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url  # Default to https if missing
    
    parsed = urlparse(url)
    scheme = "https" if parsed.scheme in ["http", "https"] else parsed.scheme
    netloc = parsed.netloc.replace("www.", "")
    sorted_query = urlencode(sorted(parse_qsl(parsed.query)))
    normalized = urlunparse((scheme, netloc, parsed.path, parsed.params, sorted_query, parsed.fragment))
    return normalized

def merge_similar_urls(urls):
    unique_urls = {}
    for url in urls:
        normalized = normalize_url(url)
        unique_urls[normalized] = url  # Keeps original URL for reference
    return list(unique_urls.values())


In [7]:
empty_keys = []

for cve_id, cve in ground_truth.items():
    # Filter out missing commits
    if not cve["commit"]:
        empty_keys.append(cve_id)
        continue
        
    # Normalize URLs
    cve["commit"] = list(merge_similar_urls(cve["commit"]))
    

# Remove empty keys
for key in empty_keys:
    del ground_truth[key]


In [8]:
len(ground_truth)

19299

In [9]:
ground_truth["CVE-2022-32202"]

{'pull': set(),
 'commit': ['github.com/thorfdbg/libjpeg/commit/51c3241b6da39df30f016b63f43f31c4011222c7'],
 'issues': {'github.com/thorfdbg/libjpeg/issues/74',
  'https://github.com/thorfdbg/libjpeg/issues/74'},
 'others': set()}

## Merge data

In [10]:
ds = {}

for cve in tqdm(cve_metadata["CVE_Items"]):
    id = cve["cve"]["CVE_data_meta"]["ID"]
    desc = cve["cve"]["description"]["description_data"][0]["value"]
    refs = cve["cve"]["references"]["reference_data"]
    published_date = cve["publishedDate"]
    ground_truth_data = ground_truth.get(id, None)
    ds[id] = {
        "published_date": published_date,
        "desc": desc,
        "refs": refs,
        "ground_truth": ground_truth_data
    }


100%|██████████| 242524/242524 [00:00<00:00, 596886.52it/s]


In [11]:
ds_gt = {k: v for k, v in ds.items() if v["ground_truth"]}
len(ds_gt)

19256

In [12]:
ds_gt["CVE-2022-32202"]

{'published_date': '2022-06-02T14:16Z',
 'desc': 'In libjpeg 1.63, there is a NULL pointer dereference in LineBuffer::FetchRegion in linebuffer.cpp.',
 'refs': [{'url': 'https://github.com/thorfdbg/libjpeg/commit/51c3241b6da39df30f016b63f43f31c4011222c7',
   'name': 'https://github.com/thorfdbg/libjpeg/commit/51c3241b6da39df30f016b63f43f31c4011222c7',
   'refsource': 'MISC',
   'tags': ['Patch', 'Third Party Advisory']},
  {'url': 'https://github.com/thorfdbg/libjpeg/issues/74',
   'name': 'https://github.com/thorfdbg/libjpeg/issues/74',
   'refsource': 'MISC',
   'tags': ['Exploit', 'Issue Tracking', 'Third Party Advisory']}],
 'ground_truth': {'pull': set(),
  'commit': ['github.com/thorfdbg/libjpeg/commit/51c3241b6da39df30f016b63f43f31c4011222c7'],
  'issues': {'github.com/thorfdbg/libjpeg/issues/74',
   'https://github.com/thorfdbg/libjpeg/issues/74'},
  'others': set()}}

## Save data

In [13]:
import pickle

with open('../data/dataset.pkl', 'wb') as f:
    pickle.dump(ds_gt, f)